In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import Row

# sample Employee info

data_employee = [
    Row(employee_id=1, name="John", city= 'New York'),
    Row(employee_id=2, name="Jane", city = 'Loss Angeles'),
    Row(employee_id=3, name="Bob", city = 'Chicago'),
    Row(employee_id=4, name="Alice", city = 'Houston'),
    Row(employee_id=5, name="Charlie", city = 'Miami')
]

# sample salary info

data_salary = [
    Row(employee_id=1, salary=50000,deprt = 'HR'),
    Row(employee_id=2, salary=60000,deprt = 'IT'),
    Row(employee_id=3, salary=70000,deprt = 'Engineering'),
    Row(employee_id=6, salary=80000,deprt = 'Marketing'),
    Row(employee_id=7, salary=90000,deprt = 'Sales'),
]

spark = SparkSession.getActiveSession()

df_employee = spark.createDataFrame(data_employee)
df_salary = spark.createDataFrame(data_salary)
# Register tempviews for SQL

# Employee info
df_employee.createOrReplaceTempView("employee")

# Salary info
df_salary.createOrReplaceTempView("salary")
#

display(df_employee)
display(df_salary)



In [0]:
df_inner = df_employee.join(df_salary, on = 'employee_id', how = 'inner')
display(df_inner)

In [0]:
%sql
Select e.employee_id, e.name, e.city, s.salary, s.deprt 
from employee e
inner join salary s
on e.employee_id = s.employee_id


### Outer Join 

An outer join returns all rows from both dataframes, martching rows where possible and filing with nulls when there is no match. This shows all employees and all salary records, even if there is no match

In [0]:
df_outer = df_employee.join(df_salary, on = 'employee_id', how ='outer')
display(df_outer)

In [0]:
%sql
select * from employee e
full outer join salary s 
on e.employee_id = s.employee_id

In [0]:
df_left_outer = df_employee.join(df_salary, on = "employee_id", how='left_outer')
display(df_left_outer)

In [0]:
%sql
SELECT e.employee_id, e.name, e.city, s.salary, s.deprt FROM employee e
Left OUTER JOIN salary s 
ON e.employee_id = s.employee_id   


In [0]:
df_right_outer = df_employee.join(df_salary, on = "employee_id", how='right_outer')
display(df_right_outer)


In [0]:
%sql
SELECT e.employee_id, e.name, e.city, s.salary, s.deprt
FROM  employee e
RIGHT OUTER JOIN salary s 
ON e.employee_id = s.employee_id

A left semi join returns only the row from the left dataframe (Employee info) where the join key exists in the right dataframe (salary info). it does not include columns from the right dataframe.

In [0]:
df_left_semi = df_employee.join(df_salary, on = "employee_id", how='left_semi')
display(df_left_semi)

In [0]:
%sql
SELECT * FROM employee e
LEFT SEMI JOIN salary s 
ON e.employee_id = s.employee_id

In [0]:
# record which are not in right side of the table
df_left_anti = df_employee.join(df_salary, on = "employee_id", how='left_anti')
display(df_left_anti)

In [0]:
%sql
SELECT * from employee e
LEFT ANTI JOIN salary s 
ON e.employee_id = s.employee_id

In [0]:
# cartesion product of two table
df_cross = df_employee.crossJoin(df_salary)
display(df_cross)

# eg we want to explore the combination of employee and salary we used cross join

In [0]:
%sql
SELECT * FROM employee CROSS JOIN salary

In [0]:
# add a duplicate row to df_employyee for demonstartion
df_employee_dup = df_employee.union(df_employee.filter(df_employee.employee_id == 1))

# remove duplicates

df_employee_nodup = df_employee_dup.dropDuplicates()

# Join and resolve column name conficts
joined = df_employee_nodup.join(df_salary,on = 'employee_id', how = 'inner').withColumnRenamed('name','employee_name')

display(joined)